<a id="speech-transcription"></a>
# VideoDB Understanding: Speech Transcription

Create timestamped words, inspect scene-aligned text, and reuse Understanding output through VideoDB transcript APIs.


<a href="https://colab.research.google.com/github/video-db/videodb-cookbook/blob/preview/guides/indexing-v2/understanding/speech-transcription/speech-transcription.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## 1. Install, connect, and choose a video

In [ ]:
!pip install -q videodb python-dotenv pandas

In [ ]:
import os
from getpass import getpass

from dotenv import load_dotenv
from videodb import connect

load_dotenv()
if not os.getenv("VIDEO_DB_API_KEY"):
    os.environ["VIDEO_DB_API_KEY"] = getpass("Enter your VideoDB API key: ")

conn = connect(api_key=os.environ["VIDEO_DB_API_KEY"])
collection = conn.get_collection()
print("Connected to VideoDB")
print("Collection:", collection.id)

In [ ]:
VIDEO_URL = "https://www.youtube.com/watch?v=vVlEVRKv4is"  # Silicon Valley - Gilfoyle is free for hire

video = collection.upload(VIDEO_URL)

# To use an existing video instead:
# VIDEO_ID = "m-..."
# video = collection.get_video(VIDEO_ID)

print("Video:", video.id)
video.play()

## 2. Run speech transcription

`spoken_words` is the public alias for `speech_transcription`. Language and speaker labels are optional.

In [ ]:
speech_run = video.understand(
    analyzers=[{
        "type": "spoken_words",
        "name": "transcript",
        "config": {
            "language": "en",
            "speaker_labels": True,
        },
    }],
    segmentation={"type": "time", "seconds": 30},
)

speech_run.wait_until_complete(timeout=1800, poll_interval=10)
print(speech_run.status)

<a id="scene-output"></a>
## 3. Scene-aligned transcript output

Every scene contains text and the overlapping timestamped words.

In [ ]:
import pandas as pd

transcript_output = speech_run.get_analyzer("transcript").get_output()
transcript_scenes = transcript_output.get("scenes", [])

pd.DataFrame([
    {
        "start": scene.get("start"),
        "end": scene.get("end"),
        "text": (scene.get("data") or {}).get("text"),
        "language": (scene.get("data") or {}).get("language"),
        "word_count": len((scene.get("data") or {}).get("words", [])),
    }
    for scene in transcript_scenes
])

<a id="words"></a>
## 4. Word timestamps and speakers

In [ ]:
words = [
    word
    for scene in transcript_scenes
    for word in (scene.get("data") or {}).get("words", [])
]

pd.DataFrame(words).head(20)

<a id="transcript-api"></a>
## 5. Reuse it through common transcript APIs

When no completed legacy transcript exists, VideoDB can resolve the completed Understanding speech analyzer.

In [ ]:
timestamped_transcript = video.get_transcript()
transcript_text = video.get_transcript_text()

print("Timestamped items:", len(timestamped_transcript))
print(transcript_text[:500])

<a id="subtitles"></a>
## 6. Optional subtitle playback

`add_subtitle()` uses the common transcript resolver, so the Understanding transcript can drive subtitle generation.

In [ ]:
GENERATE_SUBTITLES = False

if GENERATE_SUBTITLES:
    from IPython.display import display
    from videodb import play_stream

    subtitle_stream = video.add_subtitle()
    print("Subtitle stream ready")
    display(play_stream(subtitle_stream))

## Next steps

- Feed the transcript into a VLM: [Multi-analyzer pipelines](../multi-analyzer-pipelines.ipynb)
- Index and search speech: [Indexing guide](../../indexing/indexing_guide.ipynb)
- Manage pending/failed analyzers: [Outputs and operations](../outputs-and-operations.ipynb)

## Optional cleanup

In [ ]:
DELETE_RUN = False
if DELETE_RUN:
    speech_run.delete()